In [7]:
import yaml
import json
import pandas as pd

# =========================
# Paths
# =========================
yaml_path = r"D:\capstone_project\MYGRATE---Capstone-Project\original_med_dataset.yaml"
json_path = r"D:\capstone_project\MYGRATE---Capstone-Project\test\artifacts\eval-deepseek.json"

# =========================
# Load YAML
# =========================
with open(yaml_path, "r", encoding="utf-8") as f:
    yaml_data = yaml.safe_load(f)

# =========================
# Load JSON
# =========================
with open(json_path, "r", encoding="utf-8") as f:
    eval_data = json.load(f)

# =========================
# Build YAML lookup
# repo_name: DerekYRC/mini-spring
# -> mini-spring
# =========================
yaml_lookup = {}

for repo in yaml_data:
    repo_short_name = repo["repo_name"].split("/")[-1]

    yaml_lookup[repo_short_name] = repo.get(
        "test_count", {}
    ).get(
        "tests_run", 0
    )

# =========================
# Datasets
# =========================
datasets = [
    "artemis-odb",
    "balana",
    "charts4j",
    "DaisyDiff",
    "ddd-cqrs-sample",
    "dropwizard-todo",
    "druidry",
    "friendly-id",
    "geojson-jackson",
    "Jasper-report-maven-plugin",
    "java-u2flib-server",
    "jaydio",
    "jersey-jwt",
    "kafka-spout",
    "mini-spring",
    "netty-zmtp",
    "quilt",
    "rhizobia_J",
    "spark-jobs-rest-client",
    "spring-batch-rest",
    "spring-boot-rest-example",
    "spring-context-support",
    "spring-rest-exception-handler",
    "sql-to-mongo-db-query-converter",
    "suffixtree",
    "token-bucket",
    "unidecode",
    "unix4j",
    "velocity-spring-boot-project",
]

# =========================
# Build dataframe
# =========================
rows = []

for dataset in datasets:

    result = eval_data.get(dataset, {})

    json_total_tests = result.get("total_tests", 0)

    # Ưu tiên JSON
    if json_total_tests and json_total_tests > 0:
        total_unit_test = json_total_tests
    else:
        total_unit_test = yaml_lookup.get(dataset, 0)

    rows.append({
        "dataset": dataset,
        "compilation_success": result.get("compilation_success", False),
        "total_unit_test": total_unit_test,
        "passed_tests": result.get("passed_tests", 0)
    })

df = pd.DataFrame(rows)

print(df)

print("\n========== SUMMARY ==========")
print("Datasets:", len(df))
print("Compilation success:", df["compilation_success"].sum())
print("Total unit tests:", df["total_unit_test"].sum())
print("Passed tests:", df["passed_tests"].sum())

# Export
output_excel = r"D:\capstone_project\MYGRATE---Capstone-Project\test\artifacts\deepseek_summary.xlsx"
df.to_excel(output_excel, index=False)

print(f"\nSaved to: {output_excel}")

                            dataset  compilation_success  total_unit_test  \
0                       artemis-odb                 True              866   
1                            balana                 True               26   
2                          charts4j                 True              384   
3                         DaisyDiff                False              244   
4                   ddd-cqrs-sample                False               24   
5                   dropwizard-todo                False               13   
6                           druidry                 True              950   
7                       friendly-id                False               34   
8                   geojson-jackson                False               76   
9        Jasper-report-maven-plugin                 True               10   
10               java-u2flib-server                 True              230   
11                           jaydio                 True              105   